# HSV-2 genome-wide computational discovery

This researcher-facing notebook uses the v0.5 pipeline API. It defaults to a small synthetic dataset so **Run All** is safe in CI. Set `VST_REAL_DATA=1` to regenerate the cached real-data report in analysis-only mode. Computational results are not experimental or clinical evidence.

## 1. Why the previous pilot saw only UL19 and UL30
The earlier pilot was intentionally focused. It cannot establish which gene is strongest genome-wide.

## 2. What genome-wide discovery means
Every annotated gene with an eligible candidate enters the comparison.

## 3. How each gene gets an opportunity
The balanced panel combines a per-gene quota with global leaders.

## 4. Why raw candidate count is not a ranking
Long genes naturally contain more sites, so targetability uses normalized quality, fraction, uncertainty, and support views.

In [ ]:
from pathlib import Path
import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

from viral_safe_target.config import load_config
from viral_safe_target.discovery import (
    build_candidate_feature_map, gene_rank_stability, rank_genes,
    select_balanced_discovery_panel,
)

ROOT = Path.cwd()
SYNTHETIC_MODE = os.environ.get('VST_REAL_DATA', '0') != '1'
print('Mode:', 'synthetic' if SYNTHETIC_MODE else 'real cached data')

## 5. Building the balanced screening panel
One-to-many feature mappings are retained; the selection itself uses only pre-human information.

In [ ]:
if SYNTHETIC_MODE:
    features = pd.DataFrame([
        {'seqid':'SYN','feature_type':'gene','start':1,'end':45,'feature_id':'gA','name':'geneA'},
        {'seqid':'SYN','feature_type':'gene','start':35,'end':80,'feature_id':'gB','name':'geneB'},
        {'seqid':'SYN','feature_type':'gene','start':90,'end':125,'feature_id':'gC','name':'geneC'},
    ])
    rows = []
    for i, start in enumerate([1, 20, 35, 55, 90, 105, 140]):
        rows.append({'candidate_id':f'SYN-{i:02d}','reference_accession':'SYN',
          'reference_start_1based':start,'reference_end_1based':start+19,
          'strand':'-' if i % 2 else '+','guide_sequence':('ACGT'*5)[:-1]+'ACGT'[i%4],
          'pam':'TGG','pre_human_score':1-i/20,'rejection_reasons':'',
          'exact_strain_coverage':1.0,'exact_genome_count':3,'genome_count':3,
          'exact_site_accessions':'s1;s2;s3'})
    pre_candidates = pd.DataFrame(rows)
    settings = load_config(ROOT / 'configs/hsv2_pilot.yaml')
    feature_map = build_candidate_feature_map(pre_candidates, features, config=settings)
    selection = select_balanced_discovery_panel(pre_candidates, feature_map, features, top_per_gene=2, global_top=2)
    screening_panel = selection.panel
else:
    command = ['vst','discover','genome-wide','--virus','hsv2','--analysis-only']
    subprocess.run(command, cwd=ROOT, check=True)
    screening_panel = pd.read_csv(ROOT/'reports/hsv2_genome_wide/genome_wide_screening_panel.csv')
    feature_map = pd.read_csv(ROOT/'reports/hsv2_genome_wide/candidate_feature_map.csv')
    from viral_safe_target.annotations import read_gff3
    features = read_gff3(ROOT/'data/processed/hsv2_reference.gff3')
screening_panel[['candidate_id','selection_reason','mapped_gene_names']].head()

## 6. How many genes and candidates were evaluated
## 7. Ranking all genes
## 8. Did another gene rank above UL30?
## 9. Did another candidate rank above VST-240e20eb666f9c85?
Pending screens must remain pending, so these questions may correctly be not determinable.

In [ ]:
if SYNTHETIC_MODE:
    post_candidates = screening_panel.copy()
    post_candidates['screening_status'] = 'pending'
    post_candidates['post_human_rank'] = pd.NA
    post_candidates['post_human_score'] = pd.NA
    post_candidates['decision'] = 'screening_incomplete'
    for column in ['human_exact_hit_count','human_one_mismatch_hit_count',
                   'human_two_mismatch_hit_count','human_three_mismatch_hit_count',
                   'human_total_predicted_hits']:
        post_candidates[column] = pd.NA
else:
    post_candidates = pd.read_csv(ROOT/'reports/hsv2_genome_wide/genome_wide_candidates_post_human.csv')
gene_rankings = rank_genes(post_candidates, feature_map, features)
rank_stability = gene_rank_stability(post_candidates, feature_map, features)
print(f'Panel candidates: {len(post_candidates):,}; annotated gene rows: {len(gene_rankings):,}')
gene_rankings[['gene_name','screened_candidate_count','targetability_rank','confidence_level']].head(20)

## 10. Rank stability at K=10, K=25, and K=50
The nested views reuse completed results; they do not launch three searches.

## 11. Candidate funnel

In [ ]:
funnel = pd.Series({'pre-human panel':len(screening_panel),
                    'screen completed':int(post_candidates.screening_status.eq('completed').sum()),
                    'post-human ranked':int(post_candidates.post_human_rank.notna().sum())})
ax = funnel.plot.bar(title='Candidate funnel', ylabel='Candidates')
plt.tight_layout(); plt.show()
rank_stability.head(20)

## 12. Clean fraction by gene
## 13. Best candidate by gene
## 14. Targetability versus evidence coverage

In [ ]:
plot_frame = gene_rankings.set_index('gene_name')
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
clean_values = plot_frame['clean_fraction_wilson_lower_95'].dropna().head(20)
best_values = plot_frame['best_candidate_score'].dropna().head(20)
if clean_values.empty: axes[0].text(0.5, 0.5, 'Pending completed screens', ha='center')
else: clean_values.plot.bar(ax=axes[0], title='Clean fraction lower bound')
if best_values.empty: axes[1].text(0.5, 0.5, 'Pending completed screens', ha='center')
else: best_values.plot.bar(ax=axes[1], title='Best completed candidate score')
plt.tight_layout(); plt.show()
gene_rankings[['gene_name','targetability_score','evidence_coverage','biological_evidence_status']].head(20)

## 15. Newly surfaced computational candidates
## 16. Leading pair hypotheses
Pair rows are theoretical hypotheses. Cross-gene combinations are two separate sites, not one physical deletion.

## 17. What remains unknown
A zero predicted hit is not proof of safety. Accessibility, delivery, phenotype, efficacy, and biological importance require independent evidence.

## 18. What proceeds to CRISPRitz and external tools
The workflow writes a bounded deep-screening panel and pending import templates; external tools are optional.

In [ ]:
completed = post_candidates[post_candidates.screening_status.eq('completed')]
external_status = 'ready for bounded external screening' if len(completed) else 'pending Cas-OFFinder completion'
summary = {'synthetic_mode':SYNTHETIC_MODE,'completed_candidates':len(completed),
           'external_tool_status':external_status,
           'warning':'computational research output only'}
summary